# Personalized Healthcare & Medicine Recommendation System
## Machine Learning Model Training & Evaluation Notebook

This notebook documents the official machine learning pipeline for the **Personalized Healthcare & Medicine Recommendation System**:
1. **Dataset Loading**: 4,920 records with 132 binary symptom features across 41 disease classes.
2. **Data Inspection & Preprocessing**: Target encoding using `LabelEncoder`.
3. **Model Training & Comparison**:
   - **Random Forest Classifier** (`RandomForestClassifier`)
   - **Linear Support Vector Classifier** (`SVC(kernel='linear')`)
4. **Evaluation**: Classification reports, cross-validation, and accuracy comparison.
5. **Model Serialization**: Saving `rf.pkl`, `svc.pkl`, and `disease_encoder.pkl` for Flask decision-support integration.


In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
# Load the training dataset
data_path = os.path.join('..', 'data', 'raw', 'Training.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('data', 'raw', 'Training.csv')

df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape}")
print(f"Number of symptom features: {df.shape[1] - 1}")
print(f"Unique disease classes: {df['prognosis'].nunique()}")
df.head()


In [ ]:
# Separate features and target
X = df.drop('prognosis', axis=1)
y = df['prognosis']

# Fit disease label encoder
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

print(f"Total features: {len(X.columns)}")
print(f"Classes: {encoder.classes_[:5]} ... (total {len(encoder.classes_)})")


In [ ]:
# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")


In [ ]:
# 1. Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Test Accuracy: {rf_acc * 100:.2f}%")


In [ ]:
# 2. Linear Support Vector Classifier
svc_model = SVC(kernel='linear', random_state=42)
svc_model.fit(X_train, y_train)

y_pred_svc = svc_model.predict(X_test)
svc_acc = accuracy_score(y_test, y_pred_svc)
print(f"Linear SVC Test Accuracy: {svc_acc * 100:.2f}%")


In [ ]:
# Evaluation Metrics Summary
print("=== Random Forest Evaluation ===")
print(f"Accuracy: {rf_acc:.4f}")
print("\n=== Linear SVC Evaluation ===")
print(f"Accuracy: {svc_acc:.4f}")


In [ ]:
# Save models and encoder
models_dir = os.path.join('..', 'models')
os.makedirs(models_dir, exist_ok=True)

with open(os.path.join(models_dir, 'rf.pkl'), 'wb') as f:
    pickle.dump(rf_model, f)

with open(os.path.join(models_dir, 'svc.pkl'), 'wb') as f:
    pickle.dump(svc_model, f)

with open(os.path.join(models_dir, 'disease_encoder.pkl'), 'wb') as f:
    pickle.dump(encoder, f)

print("Artifacts successfully verified and serialized in models/ directory.")
